# Wildfire Inference Notebook

Loads `wildfire_model.joblib` from the training notebook output and scores new days. No `.fit()`.

**Kaggle Inputs**
1. Training notebook output (e.g. `/kaggle/input/notebooks/lakshay654/dsai-final-training`)
2. Matching history pack (`california-wildfire-knn` or `california-wildfire-median`)

**Default mode `test_year`:** rebuilds the same features on history through 2024 + `test.parquet` (2025), then scores all 2025 rows (filtered to the artifact cell subset).


## 1. Load the trained artifact

Auto-finds `wildfire_model.joblib` under the attached training output. History pack is aligned to the artifact stage (KNN ↔ knn pack, median ↔ median pack).


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "wildfire-mpl"))

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from sklearn import set_config

set_config(display="diagram")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def env_get(*names: str, default: str = "") -> str:
    for name in names:
        value = os.environ.get(name)
        if value is not None and str(value).strip() != "":
            return str(value).strip()
    return default

# Kaggle Inputs (example):
#   Training output: /kaggle/input/notebooks/lakshay654/dsai-final-training
#   KNN pack:        /kaggle/input/datasets/lakshay654/california-wildfire-knn
#   Median pack:     /kaggle/input/datasets/lakshay654/california-wildfire-median
# fire_analysis2.csv is NOT needed (selected cells are inside the artifact).

MODEL_ARTIFACT_PATH = ""  # empty = auto-find wildfire_model.joblib (or champion_model.joblib)
HISTORY_DATA_DIRECTORY = "/kaggle/input/datasets/lakshay654/california-wildfire-knn"
# Modes: test_year | replay_last_day | next_day | prepared
INFERENCE_INPUT_KIND = "test_year"
INFERENCE_INPUT_FILE = ""  # test_year: empty → HISTORY/test.parquet; next_day/prepared: set path

MODEL_ARTIFACT_PATH = env_get("WILDFIRE_MODEL_ARTIFACT", "CHAMPION_MODEL_ARTIFACT", default=MODEL_ARTIFACT_PATH)
HISTORY_DATA_DIRECTORY = env_get("WILDFIRE_HISTORY_DATA_DIR", "CHAMPION_HISTORY_DATA_DIR", default=HISTORY_DATA_DIRECTORY)
INFERENCE_INPUT_KIND = env_get("WILDFIRE_INFERENCE_KIND", "CHAMPION_INFERENCE_KIND", default=INFERENCE_INPUT_KIND)
INFERENCE_INPUT_FILE = env_get("WILDFIRE_INFERENCE_FILE", "CHAMPION_INFERENCE_FILE", default=INFERENCE_INPUT_FILE)

default_output = (
    Path("/kaggle/working/wildfire_inference_outputs")
    if Path("/kaggle/working").is_dir()
    else Path.cwd() / "notebook_outputs" / "wildfire_inference"
)
OUTPUT_DIR = Path(env_get("WILDFIRE_INFERENCE_OUTPUT_DIR", "CHAMPION_INFERENCE_OUTPUT_DIR", default=str(default_output))).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def show(value: Any) -> None:
    try:
        from IPython.display import display
        display(value)
    except ImportError:
        print(value)

def find_model_artifact() -> Path:
    configured = MODEL_ARTIFACT_PATH.strip()
    if configured:
        path = Path(configured).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"Configured model artifact does not exist: {path}")
        return path

    search_roots: list[Path] = []
    preferred = [
        Path("/kaggle/input/notebooks/lakshay654/dsai-final-training"),
        Path("/kaggle/input/dsai-final-training"),
    ]
    search_roots.extend(preferred)
    if Path("/kaggle/input").is_dir():
        search_roots.append(Path("/kaggle/input"))
    current = Path.cwd().resolve()
    search_roots.extend([current, current / "notebook_outputs"])

    names = ("wildfire_model.joblib", "champion_model.joblib")
    candidates: list[Path] = []
    for root in search_roots:
        if not root.exists():
            continue
        for name in names:
            candidates.extend(root.rglob(name))
    candidates = list(dict.fromkeys(path.resolve() for path in candidates if path.is_file()))

    # Prefer wildfire_model.joblib when both exist.
    wildfire = [p for p in candidates if p.name == "wildfire_model.joblib"]
    if len(wildfire) == 1:
        return wildfire[0]
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "wildfire_model.joblib was not found. Attach the training notebook output "
            "(e.g. /kaggle/input/notebooks/lakshay654/dsai-final-training) or set MODEL_ARTIFACT_PATH."
        )
    raise ValueError(
        "Multiple model artifacts found; set MODEL_ARTIFACT_PATH explicitly:\n"
        + "\n".join(map(str, candidates))
    )

MODEL_PATH = find_model_artifact()
artifact = joblib.load(MODEL_PATH)
required_artifact_keys = {
    "source_stage", "imputation_method", "classifier_pipeline", "ranker_pipeline",
    "probability_calibrator", "feature_columns", "base_features", "source_columns",
    "classifier_weight", "ranker_weight", "data_contract",
}
missing_artifact_keys = sorted(required_artifact_keys - set(artifact))
if missing_artifact_keys:
    raise ValueError(f"Training artifact is missing keys: {missing_artifact_keys}")

# Auto-align history pack to artifact stage if user left the default knn path but artifact is median.
_stage = artifact["source_stage"]
_default_hist = {
    "stage_c_knn": "/kaggle/input/datasets/lakshay654/california-wildfire-knn",
    "stage_c": "/kaggle/input/datasets/lakshay654/california-wildfire-median",
}
if HISTORY_DATA_DIRECTORY.strip() in {
    _default_hist["stage_c_knn"],
    _default_hist["stage_c"],
    "",
}:
    HISTORY_DATA_DIRECTORY = _default_hist.get(_stage, HISTORY_DATA_DIRECTORY)

print("=" * 72)
print("WILDFIRE INFERENCE CONFIGURATION")
print("=" * 72)
print(f"Model artifact:    {MODEL_PATH}")
print(f"Data source stage: {artifact['source_stage']}")
print(f"Cell subset:       {artifact.get('cell_subset', artifact.get('data_contract', {}).get('cell_subset', 'all'))}")
print(f"Imputation:        {artifact['imputation_method']}")
print(f"Model features:    {len(artifact['feature_columns'])}")
print(f"History directory: {HISTORY_DATA_DIRECTORY}")
print(f"Input kind:        {INFERENCE_INPUT_KIND}")
print(f"Output directory:  {OUTPUT_DIR}")
print("=" * 72)


## 2. Inspect the fitted pipelines

These objects come from training; nothing is refit here.


In [ ]:
print("Loaded classifier pipeline")
show(artifact["classifier_pipeline"])
print("Loaded ranker pipeline")
show(artifact["ranker_pipeline"])
print(f"Feature count: {len(artifact['feature_columns'])}")
print("First 20 features:", artifact["feature_columns"][:20])


## 3. Feature engineering (same as training v2)

Calendar + weather + ignition/dryness features only (no fire-history / neighbor fire features). Must match the artifact's 92/93-feature contract.


In [ ]:
def rolling_matrix(values: np.ndarray, window: int, operation: str) -> np.ndarray:
    rolling = pd.DataFrame(values.T).rolling(window=window, min_periods=1)
    return getattr(rolling, operation)().to_numpy(dtype="float32").T

def build_features(frame: pd.DataFrame, raw_base_features: list[str]):
    cells = int(frame["cell_id"].nunique())
    days = int(frame["label_date"].nunique())

    # Calendar cycles.
    day_of_year = frame["eo_asof_date"].dt.dayofyear.to_numpy(dtype="float32")
    month = frame["eo_asof_date"].dt.month.to_numpy(dtype="float32")
    calendar = pd.DataFrame({
        "day_of_year_sin": np.sin(2 * np.pi * day_of_year / 365.25),
        "day_of_year_cos": np.cos(2 * np.pi * day_of_year / 365.25),
        "month_sin": np.sin(2 * np.pi * month / 12),
        "month_cos": np.cos(2 * np.pi * month / 12),
    }, index=frame.index).astype("float32")
    frame = pd.concat([frame, calendar], axis=1)

    # Weather physics and history ending at D-5.
    temperature_c = frame["t2m_mean"].to_numpy(dtype="float64") - 273.15
    dewpoint_c = frame["d2m_mean"].to_numpy(dtype="float64") - 273.15
    saturation = 0.6108 * np.exp(17.27 * temperature_c / np.maximum(temperature_c + 237.3, 1e-6))
    actual = 0.6108 * np.exp(17.27 * dewpoint_c / np.maximum(dewpoint_c + 237.3, 1e-6))
    vpd = np.maximum(saturation - actual, 0).astype("float32")
    weather = {
        "vpd_kpa": vpd,
        "vpd_wind_interaction": vpd * frame["wind_speed_mean"].to_numpy(dtype="float32"),
        "vpd_soil_deficit_interaction": vpd * (1 - np.clip(frame["soil_moisture_index"], 0, 1)),
        "heat_soil_deficit_interaction": (
            np.maximum(frame["t2m_max"].to_numpy(dtype="float32") - 273.15, 0)
            * (1 - np.clip(frame["swvl1_mean"], 0, 1))
        ),
        "wind_gust_ratio": frame["i10fg_max"].to_numpy(dtype="float32")
        / (frame["wind_speed_mean"].to_numpy(dtype="float32") + 0.1),
    }
    rolling_specs = {
        "t2m_max": ("max",), "rh_mean": ("min",), "tp_sum_mm": ("sum",),
        "wind_speed_mean": ("max",), "i10fg_max": ("max",),
        "swvl1_mean": ("mean",), "vpd_kpa": ("max", "mean"),
    }
    arrays = {
        name: (weather[name] if name in weather else frame[name].to_numpy(dtype="float32")).reshape(cells, days)
        for name in rolling_specs
    }
    for name, operations in rolling_specs.items():
        for window in (14, 30):
            for operation in operations:
                weather[f"{name}_{operation}_{window}d"] = rolling_matrix(
                    arrays[name], window, operation
                ).reshape(-1)
    temperature = frame["t2m_max"].to_numpy(dtype="float32").reshape(cells, days)
    soil = frame["swvl1_mean"].to_numpy(dtype="float32").reshape(cells, days)
    weather["t2m_max_anomaly_30d"] = (temperature - rolling_matrix(temperature, 30, "mean")).reshape(-1)
    weather["swvl1_anomaly_30d"] = (soil - rolling_matrix(soil, 30, "mean")).reshape(-1)
    weather_frame = pd.DataFrame(weather, index=frame.index).astype("float32")
    frame = pd.concat([frame, weather_frame], axis=1)

    # Environmental ignition / dryness context only (no lagged fire labels).
    # Cell/neighbor/upwind fire-history features are omitted: they encode spread /
    # persistence and dominate next-day environmental risk scoring.
    wind = frame["wind_speed_mean"].to_numpy(dtype="float32")
    vpd = frame["vpd_kpa"].to_numpy(dtype="float32")
    soil_deficit = 1 - np.clip(frame["soil_moisture_index"].to_numpy(dtype="float32"), 0, 1)
    vegetation = np.clip(
        frame["cvh_mean"].to_numpy(dtype="float32") + frame["cvl_mean"].to_numpy(dtype="float32"), 0, 1
    )
    context = {
        "ignition_dry_windy_index": vpd * wind * soil_deficit,
        "fuel_dryness_index": vpd * soil_deficit * vegetation,
        "vpd_short_long_trend": frame["vpd_kpa_mean_14d"] - frame["vpd_kpa_mean_30d"],
    }
    context_frame = pd.DataFrame(context, index=frame.index).astype("float32")
    frame = pd.concat([frame, context_frame], axis=1)

    # The locked contract excludes a redundant source-availability flag.
    selected_base = [name for name in raw_base_features if name != "s5n_available"]
    feature_columns = list(dict.fromkeys([
        *selected_base,
        "latitude", "longitude",
        *calendar.columns,
        *weather_frame.columns,
        *context_frame.columns,
    ]))
    groups = {
        "source_after_constant_removal": len(raw_base_features),
        "source_used_by_model": len(selected_base),
        "geographic": 2,
        "calendar": len(calendar.columns),
        "weather_and_interactions": len(weather_frame.columns),
        "dryness_and_ignition_context": len(context_frame.columns),
        "total": len(feature_columns),
    }
    return frame, feature_columns, groups


## 4. Load history, build features, score

Helpers load `train+val` (or `all` ≤2024) as history, optionally append `test.parquet`, rebuild features, and score with the saved artifact.


In [ ]:
def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))

def read_table(path: str | Path) -> pd.DataFrame:
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Input file does not exist: {path}")
    suffix = path.suffix.lower()
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    raise ValueError("Input must be Parquet, CSV, JSON, JSONL, or NDJSON")

def history_candidate_roots() -> list[Path]:
    candidates: list[Path] = []
    configured = HISTORY_DATA_DIRECTORY.strip()
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.extend([
        Path("/kaggle/input/datasets/lakshay654/california-wildfire-knn"),
        Path("/kaggle/input/datasets/lakshay654/california-wildfire-median"),
        Path("/kaggle/input/california-wildfire-knn"),
        Path("/kaggle/input/california-wildfire-median"),
    ])
    if Path("/kaggle/input").is_dir():
        candidates.extend(path.parent for path in Path("/kaggle/input").rglob("dataset_metadata.json"))
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        candidates.append(base / "Milestone 5" / "kaggle_datasets" / "stage_c_knn")
        candidates.append(base / "Milestone 5" / "kaggle_datasets" / "stage_c")
        candidates.append(base / "Milestone 4" / "numerical_nextday" / "outputs" / "m4_shared_cache" / "stage_c_knn")
        candidates.append(base / "Milestone 4" / "numerical_nextday" / "outputs" / "m4_shared_cache" / "stage_c")
    return list(dict.fromkeys(path.resolve() for path in candidates))

def locate_history(expected_stage: str) -> dict[str, Path]:
    checked: list[str] = []
    for root in history_candidate_roots():
        folder = "stage_c_knn" if expected_stage == "stage_c_knn" else "stage_c"
        layouts = [
            {
                "root": root,
                "all": root / "all.parquet",
                "train": root / "train.parquet",
                "val": root / "val.parquet",
                "test": root / "test.parquet",
                "meta": root / "meta.json",
                "dataset_metadata": root / "dataset_metadata.json",
                "features": root / "feature_columns.json",
            },
            {
                "root": root / folder,
                "all": root / folder / "all.parquet",
                "train": root / folder / "train.parquet",
                "val": root / folder / "val.parquet",
                "test": root / folder / "test.parquet",
                "meta": root / folder / "meta.json",
                "dataset_metadata": root / folder / "metadata" / "dataset_metadata.json",
                "features": root / folder / "metadata" / "feature_columns.json",
            },
        ]
        for layout in layouts:
            meta_path = layout["dataset_metadata"]
            features_path = layout["features"]
            has_meta = meta_path.is_file() and features_path.is_file() and layout["meta"].is_file()
            has_table = layout["all"].is_file() or (layout["train"].is_file() and layout["val"].is_file())
            stage = read_json(meta_path).get("stage") if has_meta else None
            checked.append(f"{layout['root']} (stage={stage}, tables={has_table})")
            if has_meta and has_table and stage == expected_stage:
                return layout
    raise FileNotFoundError(
        f"No history dataset with stage={expected_stage} was found. "
        "Set HISTORY_DATA_DIRECTORY. Checked:\n" + "\n".join(f"  - {item}" for item in checked)
    )

def history_table_paths(history_info: dict[str, Path]) -> list[Path]:
    """Prefer train+val for history through 2024; else all.parquet."""
    train_p, val_p, all_p = history_info["train"], history_info["val"], history_info["all"]
    if train_p.is_file() and val_p.is_file():
        return [train_p, val_p]
    if all_p.is_file():
        return [all_p]
    raise FileNotFoundError("Need train+val.parquet or all.parquet in the history pack")

def load_parquet_tables(paths: list[Path], columns: list[str], cell_filter: list[str] | None = None) -> pd.DataFrame:
    frames = []
    for path in paths:
        if cell_filter is None:
            part = pd.read_parquet(path, columns=columns)
        else:
            part = pd.read_parquet(path, columns=columns, filters=[("cell_id", "in", cell_filter)])
        frames.append(part)
    return pd.concat(frames, ignore_index=True)

def clean_source_table(frame: pd.DataFrame, model_artifact: dict):
    frame = frame.copy()
    for name in ("feature_end_date", "eo_asof_date", "label_date"):
        frame[name] = pd.to_datetime(frame[name]).dt.normalize()
    frame = frame.sort_values(["cell_id", "label_date"]).reset_index(drop=True)
    base = model_artifact["base_features"]
    cleanup: dict[str, int] = {}

    if model_artifact["source_stage"] == "stage_c_knn":
        frame[base] = frame[base].apply(pd.to_numeric, errors="raise").astype("float32")
        flag = model_artifact["missing_flag_column"]
        flag_values = set(frame[flag].dropna().unique().tolist())
        if not flag_values.issubset({0.0, 1.0}):
            raise ValueError(f"{flag} must be binary")
        if not np.isfinite(frame[base].to_numpy(dtype="float32", copy=False)).all():
            raise ValueError("KNN source data contains NaN or infinity")
        flagged = frame[flag].eq(1)
        if not frame.loc[flagged, "s2n_available"].eq(0).all():
            raise ValueError("KNN-imputed rows must retain s2n_available=0")
        cleanup["knn_imputed_rows_flagged"] = int(flagged.sum())
    else:
        s2_invalid = frame["s2n_available"].ne(1)
        s2_values = [name for name in base if name.startswith("s2n_") and name != "s2n_available"]
        frame.loc[s2_invalid, s2_values] = np.nan
        frame.loc[s2_invalid, "s2n_available"] = 0.0
        cleanup["sentinel2_rows_marked_missing"] = int(s2_invalid.sum())
        s5_invalid = frame["s5n_available"].ne(1)
        s5_values = [name for name in base if name.startswith("s5n_") and name != "s5n_available"]
        frame.loc[s5_invalid, s5_values] = 0.0
        frame.loc[s5_invalid, "s5n_available"] = 0.0
        cleanup["sentinel5p_rows_zeroed"] = int(s5_invalid.sum())
        frame[base] = frame[base].astype("float32")

    for name in ("swvl1_mean", "swvl2_mean", "soil_moisture_index", "swvl1_mean_7d"):
        cleanup[f"{name}_negative_rows_clipped"] = int(frame[name].lt(0).sum())
        frame[name] = frame[name].clip(lower=0)
    frame["year"] = frame["label_date"].dt.year.astype("int16")
    return frame, cleanup

def validate_complete_grid(frame: pd.DataFrame) -> None:
    cells = int(frame["cell_id"].nunique())
    days = int(frame["label_date"].nunique())
    if len(frame) != cells * days or frame.duplicated(["cell_id", "label_date"]).any():
        raise ValueError("Source data must form a complete cell-by-day grid")
    if not (frame["label_date"] - frame["eo_asof_date"]).dt.days.eq(1).all():
        raise ValueError("label_date must equal eo_asof_date + 1 day")
    if not (frame["eo_asof_date"] - frame["feature_end_date"]).dt.days.eq(5).all():
        raise ValueError("eo_asof_date must equal feature_end_date + 5 days")

def artifact_cell_subset(model_artifact: dict) -> str:
    return str(
        model_artifact.get("cell_subset")
        or model_artifact.get("data_contract", {}).get("cell_subset")
        or "all"
    )

def artifact_selected_cells(model_artifact: dict) -> list | None:
    contract = model_artifact.get("data_contract", {}) or {}
    cells = contract.get("selected_cell_ids")
    if cells:
        return [str(cell) for cell in cells]
    if artifact_cell_subset(model_artifact) == "all":
        return None
    raise ValueError(
        "Artifact cell_subset is not 'all' but data_contract.selected_cell_ids is missing. "
        "Retrain with the updated training notebook."
    )

def filter_frame_to_artifact_cells(frame: pd.DataFrame, model_artifact: dict) -> pd.DataFrame:
    selected = artifact_selected_cells(model_artifact)
    if selected is None:
        return frame
    selected_set = set(selected)
    filtered = frame.loc[frame["cell_id"].astype(str).isin(selected_set)].copy()
    if filtered.empty:
        raise ValueError("No rows remain after applying the artifact cell subset")
    return filtered

def build_history_features(history_info: dict[str, Path], model_artifact: dict):
    columns = model_artifact["source_columns"]
    selected = artifact_selected_cells(model_artifact)
    paths = history_table_paths(history_info)
    history = load_parquet_tables(paths, columns, selected)
    history["label_date"] = pd.to_datetime(history["label_date"])
    history = history.loc[history["label_date"].dt.year.le(2024)].copy()
    history = filter_frame_to_artifact_cells(history, model_artifact)
    history, cleanup = clean_source_table(history, model_artifact)
    validate_complete_grid(history)
    cleanup = dict(cleanup)
    cleanup["cell_subset"] = artifact_cell_subset(model_artifact)
    cleanup["cells_after_subset"] = int(history["cell_id"].nunique())
    cleanup["history_files"] = [path.name for path in paths]
    engineered, generated_features, _ = build_features(history, model_artifact["base_features"])
    if generated_features != model_artifact["feature_columns"]:
        raise ValueError(
            "Generated feature order differs from the training artifact "
            f"(got {len(generated_features)}, expected {len(model_artifact['feature_columns'])})"
        )
    return engineered, cleanup

def prepare_test_year(
    history_info: dict[str, Path],
    model_artifact: dict,
    test_path: Path | None = None,
) -> tuple[pd.DataFrame, dict]:
    """History through 2024 + test.parquet (2025); rebuild same features; return 2025 rows."""
    columns = model_artifact["source_columns"]
    selected = artifact_selected_cells(model_artifact)
    hist_paths = history_table_paths(history_info)
    history = load_parquet_tables(hist_paths, columns, selected)
    history["label_date"] = pd.to_datetime(history["label_date"])
    history = history.loc[history["label_date"].dt.year.le(2024)].copy()

    test_file = Path(test_path) if test_path is not None else history_info["test"]
    if not test_file.is_file():
        raise FileNotFoundError(
            f"test.parquet not found at {test_file}. "
            "Attach the matching KNN/Median pack that includes test.parquet."
        )
    test = load_parquet_tables([test_file], columns, selected)
    test["label_date"] = pd.to_datetime(test["label_date"])

    history = filter_frame_to_artifact_cells(history, model_artifact)
    test = filter_frame_to_artifact_cells(test, model_artifact)

    hist_max = history["label_date"].max()
    test_min = test["label_date"].min()
    if test_min != hist_max + pd.Timedelta(days=1):
        raise ValueError(
            f"test.parquet must continue the day after history "
            f"(history ends {hist_max.date()}, test starts {test_min.date()})"
        )

    hist_cells = set(history["cell_id"].astype(str).unique())
    test_cells = set(test["cell_id"].astype(str).unique())
    if test_cells != hist_cells:
        missing = sorted(hist_cells - test_cells)[:5]
        extra = sorted(test_cells - hist_cells)[:5]
        raise ValueError(
            "test.parquet cells must match the artifact/history cell set "
            f"(missing_example={missing}, extra_example={extra})"
        )

    combined = pd.concat([history, test], ignore_index=True)
    combined, cleanup = clean_source_table(combined, model_artifact)
    validate_complete_grid(combined)
    engineered, generated_features, _ = build_features(combined, model_artifact["base_features"])
    if generated_features != model_artifact["feature_columns"]:
        raise ValueError(
            "Generated feature order differs from the training artifact "
            f"(got {len(generated_features)}, expected {len(model_artifact['feature_columns'])})"
        )

    test_dates = set(pd.to_datetime(test["label_date"]).dt.normalize())
    prepared = engineered.loc[engineered["label_date"].isin(test_dates)].copy()
    cleanup = dict(cleanup)
    cleanup["cell_subset"] = artifact_cell_subset(model_artifact)
    cleanup["history_files"] = [path.name for path in hist_paths]
    cleanup["test_file"] = test_file.name
    cleanup["test_rows"] = int(len(prepared))
    cleanup["test_days"] = int(prepared["label_date"].nunique())
    cleanup["test_cells"] = int(prepared["cell_id"].nunique())
    return prepared, cleanup

def prepare_next_day(
    new_daily_grid: pd.DataFrame,
    history_info: dict[str, Path],
    model_artifact: dict,
) -> tuple[pd.DataFrame, dict]:
    source_columns = model_artifact["source_columns"]
    required_without_target = [name for name in source_columns if name != "y_fire"]
    missing = sorted(set(required_without_target) - set(new_daily_grid.columns))
    if missing:
        raise ValueError(f"Next-day input is missing {len(missing)} columns: {missing[:15]}")
    custom = new_daily_grid[required_without_target].copy()
    for name in ("feature_end_date", "eo_asof_date", "label_date"):
        custom[name] = pd.to_datetime(custom[name]).dt.normalize()
    if custom["label_date"].nunique() != 1 or custom.duplicated(["cell_id", "label_date"]).any():
        raise ValueError("next_day input must contain one unique, duplicate-free label_date")

    selected = artifact_selected_cells(model_artifact)
    history = load_parquet_tables(history_table_paths(history_info), source_columns, selected)
    history = filter_frame_to_artifact_cells(history, model_artifact)
    for name in ("feature_end_date", "eo_asof_date", "label_date"):
        history[name] = pd.to_datetime(history[name]).dt.normalize()
    history = history.loc[history["label_date"].dt.year.le(2024)].copy()

    forecast_date = custom["label_date"].iloc[0]
    expected_date = history["label_date"].max() + pd.Timedelta(days=1)
    if forecast_date != expected_date:
        raise ValueError(f"next_day label_date must be {expected_date.date()}; received {forecast_date.date()}")
    expected_cells = set(history["cell_id"].astype(str).unique())
    custom_cells = set(custom["cell_id"].astype(str).unique())
    if custom_cells != expected_cells:
        raise ValueError(
            "next_day must contain exactly the artifact cell set "
            f"(expected {len(expected_cells)} cells for subset "
            f"{artifact_cell_subset(model_artifact)})"
        )
    archive_grid = history.sort_values("label_date").drop_duplicates("cell_id", keep="last")[["cell_id", "latitude", "longitude"]]
    coordinates = custom[["cell_id", "latitude", "longitude"]].merge(
        archive_grid, on="cell_id", how="left", suffixes=("_new", "_archive"), validate="one_to_one"
    )
    if not (
        np.allclose(coordinates["latitude_new"], coordinates["latitude_archive"], atol=1e-6)
        and np.allclose(coordinates["longitude_new"], coordinates["longitude_archive"], atol=1e-6)
    ):
        raise ValueError("next_day coordinates do not match the training grid")

    custom["y_fire"] = np.int8(0)
    combined = pd.concat([history, custom[source_columns]], ignore_index=True)
    combined, cleanup = clean_source_table(combined, model_artifact)
    validate_complete_grid(combined)
    engineered, generated_features, _ = build_features(combined, model_artifact["base_features"])
    if generated_features != model_artifact["feature_columns"]:
        raise ValueError("Generated feature order differs from the training artifact")
    prepared = engineered.loc[engineered["label_date"].eq(forecast_date)].copy()
    return prepared, cleanup

def within_day_percentile(score: np.ndarray, dates: pd.Series) -> np.ndarray:
    table = pd.DataFrame({"date": pd.to_datetime(dates).to_numpy(), "score": score, "position": np.arange(len(score))})
    table["percentile"] = table.groupby("date", sort=False)["score"].rank(method="average", pct=True)
    return table.sort_values("position")["percentile"].to_numpy(dtype=float)

def score_prepared(prepared: pd.DataFrame, model_artifact: dict) -> pd.DataFrame:
    features = model_artifact["feature_columns"]
    missing = sorted({"label_date", *features} - set(prepared.columns))
    if missing:
        raise ValueError(f"Prepared input is missing {len(missing)} columns: {missing[:15]}")
    table = prepared.copy()
    table["label_date"] = pd.to_datetime(table["label_date"]).dt.normalize()
    table[features] = table[features].apply(pd.to_numeric, errors="raise")
    if table.empty:
        raise ValueError("Inference input is empty")

    raw_probability = model_artifact["classifier_pipeline"].predict_proba(table[features])[:, 1]
    clipped = np.clip(raw_probability, 1e-7, 1 - 1e-7)
    calibrated = model_artifact["probability_calibrator"].predict_proba(
        np.log(clipped / (1 - clipped)).reshape(-1, 1)
    )[:, 1]
    sort_columns = ["label_date"] + (["cell_id"] if "cell_id" in table.columns else [])
    rank_input = table.sort_values(sort_columns)
    rank_sorted = model_artifact["ranker_pipeline"].predict(rank_input[features])
    rank_score = pd.Series(rank_sorted, index=rank_input.index).reindex(table.index).to_numpy(dtype=float)
    alert_score = (
        model_artifact["classifier_weight"] * within_day_percentile(raw_probability, table["label_date"])
        + model_artifact["ranker_weight"] * within_day_percentile(rank_score, table["label_date"])
    )
    identifiers = [name for name in (
        "feature_end_date", "eo_asof_date", "label_date", "cell_id",
        "latitude", "longitude", "y_fire",
    ) if name in table.columns]
    result = table[identifiers].copy()
    result["p_fire_raw"] = raw_probability.astype("float32")
    result["p_fire"] = calibrated.astype("float32")
    result["rank_score"] = rank_score.astype("float32")
    result["alert_score"] = alert_score.astype("float32")
    result["daily_priority_rank"] = result.groupby("label_date")["alert_score"].rank(method="first", ascending=False).astype("int32")
    result["alert_top_25"] = result["daily_priority_rank"].le(25)
    return result.sort_values(["label_date", "daily_priority_rank"]).reset_index(drop=True)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## 5. Run inference

Default: `test_year` → score all of 2025 from `test.parquet`. Alternatives: `replay_last_day`, `next_day`, `prepared`.


In [ ]:
input_kind = INFERENCE_INPUT_KIND.strip().lower()
input_path_value = INFERENCE_INPUT_FILE.strip()
cleanup_summary: dict[str, Any] = {}
history_info = None

if input_kind == "prepared":
    if not input_path_value:
        raise ValueError("INFERENCE_INPUT_FILE is required for prepared inference")
    prepared_features = read_table(input_path_value)
    inference_source = str(Path(input_path_value))
elif input_kind == "test_year":
    history_info = locate_history(artifact["source_stage"])
    test_path = Path(input_path_value).expanduser() if input_path_value else history_info["test"]
    prepared_features, cleanup_summary = prepare_test_year(history_info, artifact, test_path)
    inference_source = f"test_year from {test_path} with history through 2024"
elif input_kind in {"replay_last_day", "next_day"}:
    history_info = locate_history(artifact["source_stage"])
    if input_kind == "replay_last_day":
        engineered_history, cleanup_summary = build_history_features(history_info, artifact)
        replay_date = engineered_history["label_date"].max()
        prepared_features = engineered_history.loc[engineered_history["label_date"].eq(replay_date)].copy()
        inference_source = f"archive replay for {replay_date.date()}"
        del engineered_history
        gc.collect()
    else:
        if not input_path_value:
            raise ValueError("INFERENCE_INPUT_FILE is required for next_day inference")
        custom_grid = read_table(input_path_value)
        prepared_features, cleanup_summary = prepare_next_day(custom_grid, history_info, artifact)
        inference_source = str(Path(input_path_value))
else:
    raise ValueError("INFERENCE_INPUT_KIND must be test_year, replay_last_day, prepared, or next_day")

predictions = score_prepared(prepared_features, artifact)
predictions_path = OUTPUT_DIR / "inference_predictions.parquet"
predictions.to_parquet(predictions_path, index=False)

# Optional quick metrics when y_fire is present (e.g. 2025 test labels).
eval_summary: dict[str, Any] = {}
if "y_fire" in predictions.columns and predictions["y_fire"].notna().any():
    from sklearn.metrics import average_precision_score, roc_auc_score
    y = predictions["y_fire"].to_numpy(dtype="int8")
    p = predictions["p_fire"].to_numpy(dtype=float)
    if y.sum() > 0 and len(np.unique(y)) > 1:
        eval_summary = {
            "positives": int(y.sum()),
            "prevalence": float(y.mean()),
            "pr_auc": float(average_precision_score(y, p)),
            "roc_auc": float(roc_auc_score(y, p)),
            "recall_at_25": float(
                predictions.loc[predictions["alert_top_25"], "y_fire"].sum() / max(int(y.sum()), 1)
            ),
        }
        print("Label-aware check on scored rows:")
        show(pd.DataFrame([eval_summary]))

summary_path = OUTPUT_DIR / "inference_summary.json"
summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_artifact": str(MODEL_PATH),
    "model_sha256": sha256(MODEL_PATH),
    "source_stage": artifact["source_stage"],
    "cell_subset": artifact.get("cell_subset", artifact.get("data_contract", {}).get("cell_subset", "all")),
    "expected_grid_cells": artifact.get("data_contract", {}).get("expected_grid_cells"),
    "feature_count": len(artifact["feature_columns"]),
    "imputation_method": artifact["imputation_method"],
    "input_kind": input_kind,
    "input_source": inference_source,
    "history_root": str(history_info["root"]) if history_info else None,
    "rows": len(predictions),
    "cells": int(predictions["cell_id"].nunique()) if "cell_id" in predictions.columns else None,
    "label_dates": int(predictions["label_date"].nunique()),
    "label_date_min": str(pd.to_datetime(predictions["label_date"]).min().date()),
    "label_date_max": str(pd.to_datetime(predictions["label_date"]).max().date()),
    "cleanup": cleanup_summary,
    "eval_if_labels_present": eval_summary,
    "software": {
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__, "lightgbm": lgb.__version__,
    },
}
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")

print(f"Loaded training artifact: {MODEL_PATH}")
print(f"Inference source:         {inference_source}")
print(f"Scored rows:              {len(predictions):,}")
print(f"Label dates:              {predictions['label_date'].nunique():,}")
print(f"Features used:            {len(artifact['feature_columns'])}")
print(f"Predictions:              {predictions_path}")
print(f"Summary:                  {summary_path}")
show(predictions.head(25))


## 6. Outputs

- `/kaggle/working/wildfire_inference_outputs/inference_predictions.parquet`
- `inference_summary.json` (artifact path, dates, row counts, optional PR-AUC if labels exist)

Columns include `p_fire`, `alert_score`, `daily_priority_rank`, `alert_top_25`.
